<a href="https://colab.research.google.com/github/MAI-AIIU/deep-learning/blob/main/src/seventh-week/predicting-house-prices-mini-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Metadata
**Name**  
> Naveed Ahmad Hematmal

**Assignment**    

> Predicting House Prices | Mini Project

# Predicting House Prices on Kaggle
:label:`sec_kaggle_house`

Now that we have introduced some basic tools
for building and training deep networks
and regularizing them with techniques including
weight decay and dropout,
we are ready to put all this knowledge into practice
by participating in a Kaggle competition.
The house price prediction competition
is a great place to start.
The data is fairly generic and do not exhibit exotic structure
that might require specialized models (as audio or video might).
This dataset, collected by :citet:`De-Cock.2011`,
covers house prices in Ames, Iowa from the period 2006--2010.
It is considerably larger than the famous [Boston housing dataset](https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.names) of Harrison and Rubinfeld (1978),
boasting both more examples and more features.


In this section, we will walk you through details of
data preprocessing, model design, and hyperparameter selection.
We hope that through a hands-on approach,
you will gain some intuitions that will guide you
in your career as a data scientist.


In [1]:
%matplotlib inline
import hashlib
import os
import shutil
import stat
import tarfile
import tempfile
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

DATA_URL = 'https://d2l-data.s3-accelerate.amazonaws.com/'
DATA_DIR = Path('data')

## Downloading Data

Throughout the book, we will train and test models
on various downloaded datasets.
Here, we (**implement two utility functions**)
for downloading and extracting zip or tar files.
Again, we skip implementation details of
such utility functions.


In [2]:
def download(url, folder, sha1_hash=None):
    """Download a file into a cache directory, verifying an optional SHA-1."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    filename = Path(urllib.parse.urlparse(url).path).name
    if not filename:
        raise ValueError(f'URL does not contain a filename: {url}')
    file_path = folder / filename

    def sha1(path):
        digest = hashlib.sha1()
        with path.open('rb') as file:
            for block in iter(lambda: file.read(1 << 20), b''):
                digest.update(block)
        return digest.hexdigest()

    if file_path.is_file() and (sha1_hash is None or sha1(file_path) == sha1_hash):
        return file_path

    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(dir=folder, delete=False) as temporary_file:
            temporary_path = Path(temporary_file.name)
            with urllib.request.urlopen(url) as response:
                shutil.copyfileobj(response, temporary_file)
        if sha1_hash is not None and sha1(temporary_path) != sha1_hash:
            raise ValueError(f'SHA-1 checksum mismatch for downloaded file: {url}')
        os.replace(temporary_path, file_path)
        return file_path
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()


def extract(filename, folder):
    """Extract a ZIP or tar archive while keeping entries inside folder."""
    filename = Path(filename)
    folder = Path(folder).resolve()
    folder.mkdir(parents=True, exist_ok=True)

    def destination(member_name):
        target = (folder / member_name).resolve()
        if target != folder and folder not in target.parents:
            raise ValueError(f'Archive member escapes extraction folder: {member_name}')
        return target

    if zipfile.is_zipfile(filename):
        with zipfile.ZipFile(filename) as archive:
            for member in archive.infolist():
                target = destination(member.filename)
                mode = member.external_attr >> 16
                if stat.S_ISLNK(mode):
                    raise ValueError(f'Archive contains a symbolic link: {member.filename}')
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
        return folder

    if tarfile.is_tarfile(filename):
        with tarfile.open(filename) as archive:
            for member in archive.getmembers():
                target = destination(member.name)
                if member.isdir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                if not member.isfile():
                    raise ValueError(f'Archive contains an unsupported entry: {member.name}')
                target.parent.mkdir(parents=True, exist_ok=True)
                source = archive.extractfile(member)
                if source is None:
                    raise ValueError(f'Could not read archive member: {member.name}')
                with source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
        return folder

    raise ValueError(f'Unsupported archive format: {filename}')

## Kaggle

[Kaggle](https://www.kaggle.com) is a popular platform
that hosts machine learning competitions.
Each competition centers on a dataset and many
are sponsored by stakeholders who offer prizes
to the winning solutions.
The platform helps users to interact
via forums and shared code,
fostering both collaboration and competition.
While leaderboard chasing often spirals out of control,
with researchers focusing myopically on preprocessing steps
rather than asking fundamental questions,
there is also tremendous value in the objectivity of a platform
that facilitates direct quantitative comparisons
among competing approaches as well as code sharing
so that everyone can learn what did and did not work.
If you want to participate in a Kaggle competition,
you will first need to register for an account
(see :numref:`fig_kaggle`).

![The Kaggle website.](https://github.com/MAI-AIIU/deep-learning/blob/main/src/img/kaggle.png?raw=1)
:width:`400px`
:label:`fig_kaggle`

On the house price prediction competition page, as illustrated
in :numref:`fig_house_pricing`,
you can find the dataset (under the "Data" tab),
submit predictions, and see your ranking,
The URL is right here:

> https://www.kaggle.com/c/house-prices-advanced-regression-techniques

![The house price prediction competition page.](https://github.com/MAI-AIIU/deep-learning/blob/main/src/img/house-pricing.png?raw=1)
:width:`400px`
:label:`fig_house_pricing`

## Accessing and Reading the Dataset

Note that the competition data is separated
into training and test sets.
Each record includes the property value of the house
and attributes such as street type, year of construction,
roof type, basement condition, etc.
The features consist of various data types.
For example, the year of construction
is represented by an integer,
the roof type by discrete categorical assignments,
and other features by floating point numbers.
And here is where reality complicates things:
for some examples, some data is altogether missing
with the missing value marked simply as "na".
The price of each house is included
for the training set only
(it is a competition after all).
We will want to partition the training set
to create a validation set,
but we only get to evaluate our models on the official test set
after uploading predictions to Kaggle.
The "Data" tab on the competition tab
in :numref:`fig_house_pricing`
has links for downloading the data.

To get started, we will [**read in and process the data
using `pandas`**], which we introduced in :numref:`sec_pandas`.
For convenience, we can download and cache
the Kaggle housing dataset.
If a file corresponding to this dataset already exists in the cache directory and its SHA-1 matches `sha1_hash`, our code will use the cached file to avoid clogging up your Internet with redundant downloads.


In [3]:
class KaggleHouse:
    def __init__(self, batch_size, train=None, val=None, root=DATA_DIR):
        if (train is None) != (val is None):
            raise ValueError('train and val must either both be provided or both be omitted')
        self.batch_size = batch_size
        self.train = train
        self.val = val
        self.root = Path(root)
        if train is None:
            self.raw_train = pd.read_csv(download(
                DATA_URL + 'kaggle_house_pred_train.csv', self.root,
                sha1_hash='585e9cc93e70b39160e7921475f9bcd7d31219ce'))
            self.raw_val = pd.read_csv(download(
                DATA_URL + 'kaggle_house_pred_test.csv', self.root,
                sha1_hash='fa19780a7b011d9b009e8bff8e99922a8ee2eb90'))

The training dataset includes 1460 examples,
80 features, and one label, while the validation data
contains 1459 examples and 80 features.


In [4]:
data = KaggleHouse(batch_size=64)
print(data.raw_train.shape)
print(data.raw_val.shape)

(1460, 81)
(1459, 80)


## Data Preprocessing

Let's [**take a look at the first four and final two features
as well as the label (SalePrice)**] from the first four examples.


In [5]:
print(data.raw_train.iloc[:4, [0, 1, 2, 3, -3, -2, -1]])

   Id  MSSubClass MSZoning  LotFrontage SaleType SaleCondition  SalePrice
0   1          60       RL         65.0       WD        Normal     208500
1   2          20       RL         80.0       WD        Normal     181500
2   3          60       RL         68.0       WD        Normal     223500
3   4          70       RL         60.0       WD       Abnorml     140000


We can see that in each example, the first feature is the identifier.
This helps the model determine each training example.
While this is convenient, it does not carry
any information for prediction purposes.
Hence, we will remove it from the dataset
before feeding the data into the model.
Furthermore, given a wide variety of data types,
we will need to preprocess the data before we can start modeling.


Let's start with the numerical features.
First, we apply a heuristic,
[**replacing all missing values
by the corresponding feature's mean.**]
Then, to put all features on a common scale,
we (***standardize* the data by
rescaling features to zero mean and unit variance**):

$$x \leftarrow \frac{x - \mu}{\sigma},$$

where $\mu$ and $\sigma$ denote mean and standard deviation, respectively.
To verify that this indeed transforms
our feature (variable) such that it has zero mean and unit variance,
note that $E[\frac{x-\mu}{\sigma}] = \frac{\mu - \mu}{\sigma} = 0$
and that $E[(x-\mu)^2] = (\sigma^2 + \mu^2) - 2\mu^2+\mu^2 = \sigma^2$.
Intuitively, we standardize the data
for two reasons.
First, it proves convenient for optimization.
Second, because we do not know *a priori*
which features will be relevant,
we do not want to penalize coefficients
assigned to one feature more than any other.

[**Next we deal with discrete values.**]
These include features such as "MSZoning".
(**We replace them by a one-hot encoding**)
in the same way that we earlier transformed
multiclass labels into vectors (see :numref:`subsec_classification-problem`).
For instance, "MSZoning" assumes the values "RL" and "RM".
Dropping the "MSZoning" feature,
two new indicator features
"MSZoning_RL" and "MSZoning_RM" are created with values being either 0 or 1.
According to one-hot encoding,
if the original value of "MSZoning" is "RL",
then "MSZoning_RL" is 1 and "MSZoning_RM" is 0.
The `pandas` package does this automatically for us.


In [6]:
def preprocess_features(raw_train, raw_val):
    label = 'SalePrice'
    features = pd.concat(
        (raw_train.drop(columns=['Id', label]), raw_val.drop(columns=['Id'])))
    numeric_features = features.dtypes[features.dtypes != 'object'].index
    features[numeric_features] = features[numeric_features].apply(
        lambda column: (column - column.mean()) / column.std())
    features[numeric_features] = features[numeric_features].fillna(0)
    features = pd.get_dummies(features, dummy_na=True).astype(float)

    train_size = raw_train.shape[0]
    train = features.iloc[:train_size].copy()
    train[label] = raw_train[label].to_numpy()
    val = features.iloc[train_size:].copy()
    return train, val

You can see that this conversion increases
the number of features from 79 to 331 (excluding ID and label columns).


In [7]:
data.train, data.val = preprocess_features(data.raw_train, data.raw_val)
data.train.shape

(1460, 331)

## Error Measure

To get started we will train a linear model with squared loss. Not surprisingly, our linear model will not lead to a competition-winning submission but it does provide a sanity check to see whether there is meaningful information in the data. If we cannot do better than random guessing here, then there might be a good chance that we have a data processing bug. And if things work, the linear model will serve as a baseline giving us some intuition about how close the simple model gets to the best reported models, giving us a sense of how much gain we should expect from fancier models.

With house prices, as with stock prices,
we care about relative quantities
more than absolute quantities.
Thus [**we tend to care more about
the relative error $\frac{y - \hat{y}}{y}$**]
than about the absolute error $y - \hat{y}$.
For instance, if our prediction is off by \$100,000
when estimating the price of a house in rural Ohio,
where the value of a typical house is \$125,000,
then we are probably doing a horrible job.
On the other hand, if we err by this amount
in Los Altos Hills, California,
this might represent a stunningly accurate prediction
(there, the median house price exceeds \$4 million).

(**One way to address this problem is to
measure the discrepancy in the logarithm of the price estimates.**)
In fact, this is also the official error measure
used by the competition to evaluate the quality of submissions.
After all, a small value $\delta$ for $|\log y - \log \hat{y}| \leq \delta$
translates into $e^{-\delta} \leq \frac{\hat{y}}{y} \leq e^\delta$.
This leads to the following root-mean-squared-error between the logarithm of the predicted price and the logarithm of the label price:

$$\sqrt{\frac{1}{n}\sum_{i=1}^n\left(\log y_i -\log \hat{y}_i\right)^2}.$$


In [8]:
def get_dataloader(data, train):
    label = 'SalePrice'
    frame = data.train if train else data.val
    if label not in frame:
        return None
    features = torch.as_tensor(
        frame.drop(columns=[label]).to_numpy(dtype='float32'))
    targets = torch.log(torch.as_tensor(
        frame[label].to_numpy(dtype='float32'))).reshape(-1, 1)
    dataset = TensorDataset(features, targets)
    return DataLoader(dataset, batch_size=data.batch_size, shuffle=train)

## $K$-Fold Cross-Validation

You might recall that we introduced [**cross-validation**]
in :numref:`subsec_generalization-model-selection`, where we discussed how to deal
with model selection.
We will put this to good use to select the model design
and to adjust the hyperparameters.
We first need a function that returns
the $i^\textrm{th}$ fold of the data
in a $K$-fold cross-validation procedure.
It proceeds by slicing out the $i^\textrm{th}$ segment
as validation data and returning the rest as training data.
Note that this is not the most efficient way of handling data
and we would definitely do something much smarter
if our dataset was considerably larger.
But this added complexity might obfuscate our code unnecessarily
so we can safely omit it here owing to the simplicity of our problem.


In [9]:
def k_fold_data(data, k):
    rets = []
    fold_size = data.train.shape[0] // k
    for j in range(k):
        idx = range(j * fold_size, (j+1) * fold_size)
        rets.append(KaggleHouse(data.batch_size, data.train.drop(index=idx),
                                data.train.loc[idx]))
    return rets

[**The average validation error is returned**]
when we train $K$ times in the $K$-fold cross-validation.


In [10]:
class LinearRegression(nn.Module):
    def __init__(self, num_inputs, lr):
        super().__init__()
        self.net = nn.Linear(num_inputs, 1)
        self.lr = lr

    def forward(self, features):
        return self.net(features)


def train_model(model, train_loader, val_loader, max_epochs):
    optimizer = torch.optim.SGD(model.parameters(), lr=model.lr)
    loss_fn = nn.MSELoss()

    for _ in range(max_epochs):
        model.train()
        for features, targets in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(features), targets)
            loss.backward()
            optimizer.step()

    model.eval()
    total_loss = 0.0
    total_examples = 0
    with torch.no_grad():
        for features, targets in val_loader:
            loss = loss_fn(model(features), targets)
            total_loss += loss.item() * features.shape[0]
            total_examples += features.shape[0]
    if total_examples == 0:
        raise ValueError('Validation fold is empty')
    return total_loss / total_examples


def k_fold(data, k, lr, max_epochs=10):
    val_loss, models = [], []
    for data_fold in k_fold_data(data, k):
        num_inputs = data_fold.train.drop(columns=['SalePrice']).shape[1]
        model = LinearRegression(num_inputs, lr)
        fold_train = get_dataloader(data_fold, train=True)
        fold_val = get_dataloader(data_fold, train=False)
        val_loss.append(train_model(model, fold_train, fold_val, max_epochs))
        models.append(model)
    print(f'average validation log mse = {sum(val_loss) / len(val_loss)}')
    return models

## [**Model Selection**]

In this example, we pick an untuned set of hyperparameters
and leave it up to the reader to improve the model.
Finding a good choice can take time,
depending on how many variables one optimizes over.
With a large enough dataset,
and the normal sorts of hyperparameters,
$K$-fold cross-validation tends to be
reasonably resilient against multiple testing.
However, if we try an unreasonably large number of options
we might find that our validation
performance is no longer representative of the true error.


In [11]:
models = k_fold(data, k=5, lr=0.01, max_epochs=10)

average validation log mse = 0.17941302395846745


Notice that sometimes the number of training errors
for a set of hyperparameters can be very low,
even as the number of errors on $K$-fold cross-validation
grows considerably higher.
This indicates that we are overfitting.
Throughout training you will want to monitor both numbers.
Less overfitting might indicate that our data can support a more powerful model.
Massive overfitting might suggest that we can gain
by incorporating regularization techniques.

<!-- ##  [**Submitting Predictions on Kaggle**] -->
##  [**Submitting Predictions to the classroom**]

Now that we know what a good choice of hyperparameters should be,
we might
calculate the average predictions
on the test set
by all the $K$ models.
Saving the predictions in a csv filewill simplify uploading the results to the classroom.
<!-- will simplify uploading the results to Kaggle. -->
<!-- The following code will generate a file called `submission.csv`. -->


In [12]:
# write the code that will generate a file called submission.csv.
#Save the predictions in the csv file to simplify uploading the results to the classroom
test_features = torch.tensor(
    data.val.to_numpy(dtype='float32')
)

# Average predictions from all 5 K-fold models
with torch.no_grad():
    predictions = torch.stack([
        model(test_features).reshape(-1)
        for model in models
    ]).mean(dim=0)

# Convert log predictions back to house prices
predictions = torch.exp(predictions).numpy()

# Create submission.csv
submission = pd.DataFrame({
    'Id': data.raw_val['Id'],
    'SalePrice': predictions
})

submission.to_csv('submission.csv', index=False)


<!-- Next, as demonstrated in :numref:`fig_kaggle_submit2`,
we can submit our predictions on Kaggle
and see how they compare with the actual house prices (labels)
on the test set.
The steps are quite simple:

* Log in to the Kaggle website and visit the house price prediction competition page.
* Click the “Submit Predictions” or “Late Submission” button.
* Click the “Upload Submission File” button in the dashed box at the bottom of the page and select the prediction file you wish to upload.
* Click the “Make Submission” button at the bottom of the page to view your results. -->

<!-- ![Submitting data to Kaggle](../img/kaggle-submit2.png)
:width:`400px`
:label:`fig_kaggle_submit2` -->

## Summary and Discussion

Real data often contains a mix of different data types and needs to be preprocessed.
Rescaling real-valued data to zero mean and unit variance is a good default. So is replacing missing values with their mean.
Furthermore, transforming categorical features into indicator features allows us to treat them like one-hot vectors.
When we tend to care more about
the relative error than about the absolute error,
we can
measure the discrepancy in the logarithm of the prediction.
To select the model and adjust the hyperparameters,
we can use $K$-fold cross-validation .



## Exercises

<!-- 1. Submit your predictions for this section to Kaggle. How good are they? -->
1. write the code that will generate a file called submission.csv in the last code cell and save the predictions in the generated csv file.

1. Submit your predictions saved in the csv fle for this section along with the lab to classroom. How good are they?
1. Is it always a good idea to replace missing values by a mean? Hint: can you construct a situation where the values are not missing at random?
1. Improve the score by tuning the hyperparameters through $K$-fold cross-validation.
1. Improve the score by improving the model (e.g., layers, weight decay, and dropout).
1. What happens if we do not standardize the continuous numerical features as we have done in this section?


[Discussions](https://discuss.d2l.ai/t/107)


For all above six problems, solution is below with proper comments to diffrnt them.

In [13]:


import numpy as np
import pandas as pd
import torch
from torch import nn

torch.manual_seed(42)

# ============================================================
# Exercise 1: Generate submission.csv using K-fold models
# ============================================================

test_features = torch.tensor(
    data.val.to_numpy(dtype="float32")
)


with torch.no_grad():
    log_predictions = torch.stack([
        model(test_features).reshape(-1)
        for model in models
    ]).mean(dim=0)

predictions = torch.exp(log_predictions).numpy()

submission = pd.DataFrame({
    "Id": data.raw_val["Id"],
    "SalePrice": predictions
})

submission.to_csv("submission.csv", index=False)
print("1. submission.csv generated successfully!")


# ============================================================
# Exercise 2: Submit predictions and evaluate performance
# ============================================================

print("\n2. Submission information:")
print(f"Total predictions: {len(submission)}")
print(submission.head())
print("Upload submission.csv with your notebook to classroom.")
print("Actual test score requires ground-truth labels or a Kaggle submission.")


# ============================================================
# Exercise 3: Is replacing missing values with mean always good?
# ============================================================

print("\n3. Missing values explanation:")
print("No, mean imputation is not always appropriate.")
print("Example: Expensive houses may omit income information.")
print("Missingness may therefore depend on the actual value.")
print("Mean imputation can introduce bias.")
print("Alternatives: median, missing-value indicators, or model-based imputation.")


# ============================================================
# Helper functions for Exercises 4, 5, and 6
# ============================================================

X = torch.tensor(
    data.train.drop(columns=["SalePrice"]).to_numpy(dtype="float32")
)

y = torch.log(torch.tensor(
    data.train["SalePrice"].to_numpy(dtype="float32")
)).reshape(-1, 1)

X_test = test_features


def build_model(num_features, hidden=False, dropout=0.0):
    if hidden:
        return nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    return nn.Linear(num_features, 1)


def fit_model(model, X_train, y_train, lr, epochs, weight_decay=0):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        for indices in torch.randperm(len(X_train)).split(64):
            features = X_train[indices]
            targets = y_train[indices]

            optimizer.zero_grad()
            loss = loss_fn(model(features), targets)
            loss.backward()
            optimizer.step()

    return model


def cross_validate(X_data, y_data, lr=0.001, epochs=30,
                   hidden=False, dropout=0.0,
                   weight_decay=0.0, k=5):

    indices = torch.randperm(len(X_data), generator=torch.Generator().manual_seed(42))
    folds = torch.tensor_split(indices, k)
    scores = []

    for i in range(k):
        val_idx = folds[i]
        train_idx = torch.cat([folds[j] for j in range(k) if j != i])

        model = build_model(X_data.shape[1], hidden, dropout)
        fit_model(model, X_data[train_idx], y_data[train_idx],
                  lr, epochs, weight_decay)

        model.eval()
        with torch.no_grad():
            predicted = model(X_data[val_idx])
            rmse = torch.sqrt(
                torch.mean((predicted - y_data[val_idx]) ** 2)
            ).item()

        scores.append(rmse)

    return np.mean(scores)


# ============================================================
# Exercise 4: Tune hyperparameters using K-fold validation
# ============================================================

print("\n4. Hyperparameter tuning:")

best_score = float("inf")
best_params = None

for lr in [0.01, 0.001, 0.0001]:
    for epochs in [20, 50]:
        score = cross_validate(X, y, lr=lr, epochs=epochs)
        print(f"lr={lr}, epochs={epochs}, RMSE={score:.4f}")

        if score < best_score:
            best_score = score
            best_params = (lr, epochs)

print(f"Best parameters: lr={best_params[0]}, epochs={best_params[1]}")
print(f"Best linear validation RMSE: {best_score:.4f}")


# ============================================================
# Exercise 5: Improve model with layers, dropout, weight decay
# ============================================================

print("\n5. Improving the model:")

improved_score = cross_validate(
    X, y,
    lr=0.001,
    epochs=50,
    hidden=True,
    dropout=0.1,
    weight_decay=0.001
)

print(f"Linear model RMSE: {best_score:.4f}")
print(f"Improved neural network RMSE: {improved_score:.4f}")

if improved_score < best_score:
    print("Neural network performed better.")
    best_model = build_model(X.shape[1], hidden=True, dropout=0.1)
    final_lr, final_epochs, final_wd = 0.001, 50, 0.001
else:
    print("Linear model performed better.")
    best_model = build_model(X.shape[1])
    final_lr, final_epochs = best_params
    final_wd = 0.0

# Train selected model on all training data
fit_model(best_model, X, y, final_lr, final_epochs, final_wd)
best_model.eval()

with torch.no_grad():
    final_predictions = torch.exp(
        best_model(X_test).reshape(-1)
    ).numpy()

# Update submission.csv with selected model
submission = pd.DataFrame({
    "Id": data.raw_val["Id"],
    "SalePrice": final_predictions
})

submission.to_csv("submission.csv", index=False)
print("submission.csv updated using selected model.")


# ============================================================
# Exercise 6: What happens without feature standardization?
# ============================================================

print("\n6. Training without standardization:")

# Rebuild features without numerical standardization
raw_features = pd.concat([
    data.raw_train.drop(columns=["Id", "SalePrice"]),
    data.raw_val.drop(columns=["Id"])
])

numeric_cols = raw_features.select_dtypes(
    include=[np.number]
).columns

raw_features[numeric_cols] = raw_features[numeric_cols].fillna(
    raw_features[numeric_cols].mean()
)

raw_features = pd.get_dummies(
    raw_features, dummy_na=True
).astype("float32")

X_unscaled = torch.tensor(
    raw_features.iloc[:len(data.raw_train)].to_numpy(),
    dtype=torch.float32
)

unscaled_score = cross_validate(
    X_unscaled, y,
    lr=0.0001,
    epochs=50
)

print(f"Standardized RMSE: {best_score:.4f}")
print(f"Non-standardized RMSE: {unscaled_score:.4f}")

print("Without standardization:")
print("- Features have different numerical scales.")
print("- Gradient-based optimization can become unstable.")
print("- Large-valued features can dominate updates.")
print("- Training may converge slowly or produce NaN losses.")
print("- Standardization generally improves optimization.")

print("\nAll 6 exercises completed!")
print("Final submission file: submission.csv")


1. submission.csv generated successfully!

2. Submission information:
Total predictions: 1459
     Id      SalePrice
0  1461   87373.296875
1  1462  442385.125000
2  1463  193373.265625
3  1464  249445.156250
4  1465  181243.093750
Upload submission.csv with your notebook to classroom.
Actual test score requires ground-truth labels or a Kaggle submission.

3. Missing values explanation:
No, mean imputation is not always appropriate.
Example: Expensive houses may omit income information.
Missingness may therefore depend on the actual value.
Mean imputation can introduce bias.
Alternatives: median, missing-value indicators, or model-based imputation.

4. Hyperparameter tuning:
lr=0.01, epochs=20, RMSE=0.1837
lr=0.01, epochs=50, RMSE=0.1530
lr=0.001, epochs=20, RMSE=1.6177
lr=0.001, epochs=50, RMSE=0.2968
lr=0.0001, epochs=20, RMSE=10.4118
lr=0.0001, epochs=50, RMSE=8.3011
Best parameters: lr=0.01, epochs=50
Best linear validation RMSE: 0.1530

5. Improving the model:
Linear model RMSE: 0

All reasons and answers are also I put in above cell inside `print` functions, so running also print them